## Étape 8 : valider les règles de décision sur les tickets simulés

Ce notebook fait passer les 10 tickets de la table `TICKETS` dans le graphe réel, puis compare la décision obtenue à la décision attendue (`EXPECTED_DECISION`).

Il ne modifie aucune table. Chaque ticket coûte quelques appels au LLM.

Après avoir modifié un fichier de `src/`, redémarre le noyau avant de relancer.

In [ ]:
!pip install langgraph langchain-openai rank_bm25

In [ ]:
import os
import sys

import pandas as pd
from snowflake.snowpark.context import get_active_session

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.append(ROOT)

from src.graph import build_graph
from src.llm import make_llm
from src.trace import show_trace

session = get_active_session()
llm = make_llm()
graph = build_graph(llm=llm, session=session)

pd.set_option("display.max_colwidth", 90)

### 1. Les tickets simulés

In [ ]:
tickets = session.table("PROJECT_DB.PUBLIC.TICKETS").to_pandas().sort_values("TICKET_ID").reset_index(drop=True)
print(len(tickets), "tickets")
tickets[["TICKET_ID", "QUOTE_ID", "BOOKING_ID", "CREATED_DATE", "CASE_TYPE", "EXPECTED_DECISION"]]

### 2. Du ticket de la table au ticket du graphe

La table `TICKETS` ne contient que l'identifiant, la date, la description et les références de devis et de réservation. Les autres champs du formulaire (route, pricelist, référence de ligne) sont vides. On passe le ticket au graphe tel qu'il est, sans rien ajouter.

In [ ]:
def text(value):
    return "" if value is None or pd.isna(value) else str(value)


def date_text(value):
    if value is None or pd.isna(value):
        return ""
    return pd.to_datetime(value).strftime("%Y-%m-%d")


def to_graph_ticket(row):
    created = date_text(row["CREATED_DATE"])
    return {
        "ticket_id": row["TICKET_ID"],
        "received_at": created,
        "time_of_issue": created,
        "email": "",
        "description": text(row["DESCRIPTION"]),
        "pricelist_number": "",
        "pricing_group": "",
        "pol": "",
        "pod": "",
        "container_type": "",
        "line_reference": "",
        "quote_id": text(row["QUOTE_ID"]),
        "booking_id": text(row["BOOKING_ID"]),
    }


to_graph_ticket(tickets.iloc[0])

### 3. Passage des 10 tickets dans le graphe

In [ ]:
rows, states = [], {}
for _, row in tickets.iterrows():
    tid = row["TICKET_ID"]
    try:
        state = graph.invoke({"ticket": to_graph_ticket(row), "trace": []})
    except Exception as e:
        rows.append({"ticket": tid, "type de cas": row["CASE_TYPE"], "attendu": row["EXPECTED_DECISION"],
                     "obtenu": "ERREUR", "règle": f"{type(e).__name__}: {e}"[:120], "OK": False})
        continue
    states[tid] = state
    data = state.get("data") or {}
    rows.append({
        "ticket": tid,
        "type de cas": row["CASE_TYPE"],
        "attendu": row["EXPECTED_DECISION"],
        "obtenu": state["decision"],
        "règle": state["decision_rule"],
        "cas de prix": state.get("price_case"),
        "lignes candidates": data.get("nb_candidates"),
        "preuve": data.get("evidence"),
        "source fiable": state.get("docs_reliable"),
        "équipe": state.get("team"),
        "parcours": " > ".join(e["noeud"] for e in state["trace"]),
        "OK": state["decision"] == row["EXPECTED_DECISION"],
    })

bilan = pd.DataFrame(rows)
print(f"{int(bilan['OK'].sum())} / {len(bilan)} décisions conformes")
bilan

### 4. Répartition par type de cas

In [ ]:
bilan.groupby("type de cas")["OK"].agg(["sum", "count"]).rename(columns={"sum": "conformes", "count": "tickets"})

### 5. Le détail d'un ticket

Change l'identifiant pour lire la trace et le texte produit pour un autre ticket.

In [ ]:
TICKET = "TK-SIM-001"

state = states[TICKET]
print("TICKET :", state["ticket"]["description"])
show_trace(state["trace"])
print("\nISSUE :", state["decision"], "| règle :", state["decision_rule"])
print("RAISON :", state["decision_reason"])
print("\n" + state["answer"])